In [ ]:
# Fabric Notebook: NB_10_BRONZE_TO_SILVER_PREPROCESS
# Default Lakehouse: lh_vv_silver (MANDATORY)
# READ : Bronze poi_raw_event
# WRITE: Silver _stage_raw_event, _stage_poi_language, poi_source_map
# Core responsibility: one raw event -> N language-specific logical POIs.

from pyspark.sql import functions as F
from pyspark.sql.types import *

# PARAMETERS - mark as Fabric parameter cell
run_id = globals().get("run_id", "DEV_RUN")
watermark_from = globals().get("watermark_from", "1901-01-01 00:00:00")
watermark_to = globals().get("watermark_to", "2999-12-31 00:00:00")

BRONZE_RAW = "sgr_visitvn_dev.lh_vv_bronze.dbo.poi_raw_event"
STAGE_RAW = "_stage_raw_event"
STAGE_LANG = "_stage_poi_language"
SOURCE_MAP = "poi_source_map"
GOLD_POI_MASTER = "sgr_visitvn_dev.lh_vv_gold.dbo.poi_master"

# ---------- Helper: current deterministic ID formatting ----------
def uuidish_md5_sql(expr: str) -> str:
    return f"regexp_replace(md5({expr}), '(.{{8}})(.{{4}})(.{{4}})(.{{4}})(.{{12}})', '$1-$2-$3-$4-$5')"

# ---------- 1. Create run-scoped staging tables ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STAGE_RAW} (
  run_id STRING,
  event_id STRING,
  source_name STRING,
  source_id STRING,
  crawled_at TIMESTAMP,
  root_language_code STRING,
  normalized_payload STRING,
  poi_name STRING,
  poi_name_normalized STRING,
  business_sector STRING,
  business_category STRING,
  types_json STRING,
  operating_status STRING,
  slug STRING,
  partner_id STRING,
  address_json STRING,
  contact_json STRING,
  price_json STRING,
  content_json STRING,
  rating_json STRING,
  review_json STRING,
  media_json STRING,
  opening_hours_json STRING,
  policy_json STRING,
  amenity_json STRING,
  raw_data_json STRING,
  enrichment_json STRING,
  enrichment_siblings_json STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STAGE_LANG} (
  run_id STRING,
  event_id STRING,
  source_name STRING,
  source_id STRING,
  crawled_at TIMESTAMP,
  lang STRING,
  poi_id STRING,
  poi_group_id STRING,
  is_root_language BOOLEAN,
  poi_name STRING,
  poi_name_normalized STRING,
  business_sector STRING,
  business_category STRING,
  types_json STRING,
  operating_status STRING,
  slug STRING,
  partner_id STRING,
  address_json STRING,
  contact_json STRING,
  price_json STRING,
  content_json STRING,
  rating_json STRING,
  review_json STRING,
  media_json STRING,
  opening_hours_json STRING,
  policy_json STRING,
  amenity_json STRING,
  raw_data_json STRING,
  localized_enrichment_json STRING,
  source_enrichment_poi_id STRING,
  enrichment_id STRING,
  enrichment_lang STRING,
  first_seen_at TIMESTAMP,
  last_seen_at TIMESTAMP,
  rn_by_poi INT,
  created_at TIMESTAMP
) USING DELTA
""")

# Rerun safety for same run_id
spark.sql(f"DELETE FROM {STAGE_RAW} WHERE run_id = '{run_id}'")
spark.sql(f"DELETE FROM {STAGE_LANG} WHERE run_id = '{run_id}'")

# ---------- 2. Stage raw: parse only top-level/scalar fields and preserve component JSON ----------
spark.sql(f"""
INSERT INTO {STAGE_RAW}
SELECT
  '{run_id}' AS run_id,
  CAST(event_id AS STRING),
  source_name,
  source_id,
  crawled_at,
  lower(trim(language_code)) AS root_language_code,
  normalized_payload,
  get_json_object(normalized_payload, '$.poi_name') AS poi_name,
  get_json_object(normalized_payload, '$.poi_name_normalized') AS poi_name_normalized,
  get_json_object(normalized_payload, '$.business_sector') AS business_sector,
  get_json_object(normalized_payload, '$.business_category') AS business_category,
  get_json_object(normalized_payload, '$.types') AS types_json,
  get_json_object(normalized_payload, '$.operating_status') AS operating_status,
  get_json_object(normalized_payload, '$.slug') AS slug,
  get_json_object(normalized_payload, '$.partner_id') AS partner_id,
  get_json_object(normalized_payload, '$.poi_address') AS address_json,
  get_json_object(normalized_payload, '$.poi_contact') AS contact_json,
  get_json_object(normalized_payload, '$.poi_price') AS price_json,
  get_json_object(normalized_payload, '$.poi_content') AS content_json,
  get_json_object(normalized_payload, '$.poi_rating') AS rating_json,
  get_json_object(normalized_payload, '$.poi_review') AS review_json,
  get_json_object(normalized_payload, '$.poi_media') AS media_json,
  get_json_object(normalized_payload, '$.poi_opening_hours') AS opening_hours_json,
  get_json_object(normalized_payload, '$.policies') AS policy_json,
  get_json_object(normalized_payload, '$.poi_amenity') AS amenity_json,
  get_json_object(normalized_payload, '$.raw_data') AS raw_data_json,
  get_json_object(normalized_payload, '$.extra_info.enrichment') AS enrichment_json,
  get_json_object(normalized_payload, '$.extra_info.enrichmentSiblings') AS enrichment_siblings_json,
  current_timestamp()
FROM {BRONZE_RAW}
WHERE normalized_payload IS NOT NULL
  AND source_name IS NOT NULL
  AND source_id IS NOT NULL
  AND crawled_at > TIMESTAMP('{watermark_from}')
  AND crawled_at <= TIMESTAMP('{watermark_to}')
""")

# ---------- 3. Expand dynamic languages ----------
# Spark SQL MAP<STRING,STRING> lets us explode arbitrary sibling keys without hardcoding en/ko/...
poi_id_expr = uuidish_md5_sql("concat(source_name, source_id, lang)")
group_id_expr = uuidish_md5_sql("concat(source_name, source_id)")

spark.sql(f"""
INSERT INTO {STAGE_LANG}
WITH base AS (
  SELECT
    *,
    from_json(enrichment_siblings_json, 'MAP<STRING,STRING>') AS sibling_map,
    lower(trim(get_json_object(enrichment_json, '$.lang'))) AS primary_enrichment_lang
  FROM {STAGE_RAW}
  WHERE run_id = '{run_id}'
),
lang_rows AS (
  -- root language
  SELECT *, root_language_code AS lang,
         CASE WHEN root_language_code = primary_enrichment_lang THEN enrichment_json ELSE element_at(sibling_map, root_language_code) END AS localized_enrichment_json
  FROM base WHERE root_language_code IS NOT NULL AND root_language_code <> ''

  UNION ALL

  -- primary enrichment language (in case root language is missing/different)
  SELECT *, primary_enrichment_lang AS lang, enrichment_json AS localized_enrichment_json
  FROM base WHERE primary_enrichment_lang IS NOT NULL AND primary_enrichment_lang <> ''

  UNION ALL

  -- dynamic sibling languages
  SELECT b.*, lower(trim(kv.key)) AS lang, kv.value AS localized_enrichment_json
  FROM base b
  LATERAL VIEW explode(b.sibling_map) kv AS key, value
),
normalized_lang_rows AS (
  SELECT
    *,
    lower(trim(lang)) AS normalized_lang
  FROM lang_rows
  WHERE lang IS NOT NULL AND trim(lang) <> ''
),
event_language_validation AS (
  SELECT
    event_id,
    source_name,
    source_id,
    MAX(CASE WHEN normalized_lang = 'vi' THEN 1 ELSE 0 END) AS has_vi,
    MAX(CASE WHEN normalized_lang = 'en' THEN 1 ELSE 0 END) AS has_en,
    MAX(CASE WHEN normalized_lang = 'ko' THEN 1 ELSE 0 END) AS has_ko
  FROM normalized_lang_rows
  GROUP BY event_id, source_name, source_id
),
valid_events AS (
  SELECT event_id, source_name, source_id
  FROM event_language_validation
  WHERE has_vi = 1
    AND has_en = 1
    AND has_ko = 1
),
dedup_lang AS (
  SELECT
    nlr.*,
    ROW_NUMBER() OVER (
      PARTITION BY nlr.event_id, nlr.source_name, nlr.source_id, nlr.normalized_lang
      ORDER BY
        CASE WHEN nlr.localized_enrichment_json IS NOT NULL THEN 0 ELSE 1 END,
        nlr.crawled_at DESC
    ) AS lang_rn
  FROM normalized_lang_rows nlr
  INNER JOIN valid_events ve
    ON ve.event_id = nlr.event_id
   AND ve.source_name = nlr.source_name
   AND ve.source_id = nlr.source_id
),
expanded AS (
  SELECT
    run_id, event_id, source_name, source_id, crawled_at, lang,
    {poi_id_expr} AS poi_id,
    {group_id_expr} AS poi_group_id,
    (lang = root_language_code) AS is_root_language,
    -- Prefer localized enrichment display fields; otherwise shared root fields.
    COALESCE(get_json_object(localized_enrichment_json, '$.poiName'), poi_name) AS poi_name,
    CASE
      WHEN get_json_object(localized_enrichment_json, '$.poiName') IS NOT NULL
        THEN lower(trim(get_json_object(localized_enrichment_json, '$.poiName')))
      ELSE poi_name_normalized
    END AS poi_name_normalized,
    COALESCE(get_json_object(localized_enrichment_json, '$.refBusinessSector'), business_sector) AS business_sector,
    business_category,
    types_json,
    operating_status,
    slug,
    partner_id,
    address_json, contact_json, price_json, content_json, rating_json, review_json,
    media_json, opening_hours_json, policy_json, amenity_json, raw_data_json,
    localized_enrichment_json,
    get_json_object(localized_enrichment_json, '$.poiId') AS source_enrichment_poi_id,
    get_json_object(localized_enrichment_json, '$.enrichmentId') AS enrichment_id,
    lower(trim(get_json_object(localized_enrichment_json, '$.lang'))) AS enrichment_lang,
    crawled_at AS first_seen_at,
    crawled_at AS last_seen_at
  FROM dedup_lang
  WHERE lang_rn = 1
    AND lower(trim(lang)) IN ('vi','en','ko')
),
ranked AS (
  SELECT *,
    MIN(crawled_at) OVER (PARTITION BY source_name, source_id, lang) AS batch_first_seen_at,
    MAX(crawled_at) OVER (PARTITION BY source_name, source_id, lang) AS batch_last_seen_at,
    ROW_NUMBER() OVER (PARTITION BY poi_id ORDER BY crawled_at DESC, event_id DESC) AS rn_by_poi
  FROM expanded
)
SELECT
  run_id, event_id, source_name, source_id, crawled_at, lang, poi_id, poi_group_id,
  is_root_language, poi_name, poi_name_normalized, business_sector, business_category,
  types_json, operating_status, slug, partner_id,
  address_json, contact_json, price_json, content_json, rating_json, review_json, media_json,
  opening_hours_json, policy_json, amenity_json, raw_data_json,
  localized_enrichment_json, source_enrichment_poi_id, enrichment_id, enrichment_lang,
  batch_first_seen_at, batch_last_seen_at, rn_by_poi, current_timestamp()
FROM ranked
""")


# ---------- 3A. Skip existing POIs explicitly marked is_enrichable = false ----------
# Only explicit false is skipped. true / NULL / missing Gold row continue normally.
# The gate is applied after poi_id resolution, therefore each language-specific POI
# from the same Bronze event is evaluated independently.
_skip_not_enrichable = spark.sql(f"""
SELECT COUNT(*) AS c
FROM {STAGE_LANG} s
INNER JOIN {GOLD_POI_MASTER} pm ON pm.poi_id = s.poi_id
WHERE s.run_id = '{run_id}' AND pm.is_enrichable = false
""").first()["c"]

# Fabric Delta DELETE does not support the correlated EXISTS subquery used here.
# Materialize the blocked poi_id set as a temp view, then use Delta MERGE ... DELETE.
# This keeps the same business semantics while remaining Fabric/Delta compatible.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_not_enrichable_poi AS
SELECT DISTINCT pm.poi_id
FROM {GOLD_POI_MASTER} pm
INNER JOIN {STAGE_LANG} s
  ON s.poi_id = pm.poi_id
WHERE s.run_id = '{run_id}'
  AND pm.is_enrichable = false
""")

spark.sql(f"""
MERGE INTO {STAGE_LANG} tgt
USING v2_not_enrichable_poi src
ON  tgt.poi_id = src.poi_id
AND tgt.run_id = '{run_id}'
WHEN MATCHED THEN DELETE
""")

_processable_after_enrichable = spark.sql(
    f"SELECT COUNT(*) AS c FROM {STAGE_LANG} WHERE run_id = '{run_id}'"
).first()["c"]

print("NB10 is_enrichable gate:", {
    "skipped_not_enrichable_count": _skip_not_enrichable,
    "processable_language_poi_count": _processable_after_enrichable,
})

# ---------- 3B. Required-language validation summary ----------
# Business rule:
#   A raw event is eligible only when it contains ALL required languages: vi, en, ko.
#   Events missing any required language are skipped individually and DO NOT fail the batch.
#   Only valid events were inserted into _stage_poi_language above.
language_validation_summary = spark.sql(f"""
WITH base AS (
  SELECT
    event_id,
    source_name,
    source_id,
    lower(trim(root_language_code)) AS root_language_code,
    lower(trim(get_json_object(enrichment_json, '$.lang'))) AS primary_enrichment_lang,
    from_json(enrichment_siblings_json, 'MAP<STRING,STRING>') AS sibling_map
  FROM {STAGE_RAW}
  WHERE run_id = '{run_id}'
),
langs AS (
  SELECT event_id, source_name, source_id, root_language_code AS lang
  FROM base
  WHERE root_language_code IS NOT NULL AND root_language_code <> ''

  UNION ALL

  SELECT event_id, source_name, source_id, primary_enrichment_lang AS lang
  FROM base
  WHERE primary_enrichment_lang IS NOT NULL AND primary_enrichment_lang <> ''

  UNION ALL

  SELECT
    b.event_id,
    b.source_name,
    b.source_id,
    lower(trim(kv.key)) AS lang
  FROM base b
  LATERAL VIEW explode(b.sibling_map) kv AS key, value
),
validation AS (
  SELECT
    event_id,
    source_name,
    source_id,
    MAX(CASE WHEN lang = 'vi' THEN 1 ELSE 0 END) AS has_vi,
    MAX(CASE WHEN lang = 'en' THEN 1 ELSE 0 END) AS has_en,
    MAX(CASE WHEN lang = 'ko' THEN 1 ELSE 0 END) AS has_ko
  FROM langs
  WHERE lang IS NOT NULL AND lang <> ''
  GROUP BY event_id, source_name, source_id
)
SELECT
  COUNT(*) AS evaluated_event_count,
  SUM(CASE WHEN has_vi = 1 AND has_en = 1 AND has_ko = 1 THEN 1 ELSE 0 END) AS valid_event_count,
  SUM(CASE WHEN NOT (has_vi = 1 AND has_en = 1 AND has_ko = 1) THEN 1 ELSE 0 END) AS skipped_event_count
FROM validation
""").first()

print(
    "NB10 required-language validation:",
    {
        "evaluated_event_count": language_validation_summary["evaluated_event_count"],
        "valid_event_count": language_validation_summary["valid_event_count"],
        "skipped_event_count": language_validation_summary["skipped_event_count"],
    },
)

# ---------- 4. Persistent source map MERGE: NEVER TRUNCATE ----------
# Add poi_group_id if approved/needed. Safe to run once manually if ALTER conflicts on existing column.
try:
    spark.sql(f"ALTER TABLE {SOURCE_MAP} ADD COLUMNS (poi_group_id STRING)")
except Exception:
    pass

spark.sql(f"""
MERGE INTO {SOURCE_MAP} AS tgt
USING (
  SELECT
    source_name,
    source_id,
    lang AS language_code,
    poi_id,
    poi_group_id,
    MIN(first_seen_at) AS first_seen_at,
    MAX(last_seen_at) AS last_seen_at,
    max_by(event_id, crawled_at) AS event_id
  FROM {STAGE_LANG}
  WHERE run_id = '{run_id}'
  GROUP BY source_name, source_id, lang, poi_id, poi_group_id
) src
ON tgt.source_name = src.source_name
AND tgt.source_id = src.source_id
AND tgt.language_code = src.language_code
WHEN MATCHED THEN UPDATE SET
  tgt.poi_id = src.poi_id,
  tgt.poi_group_id = src.poi_group_id,
  tgt.last_seen_at = greatest(tgt.last_seen_at, src.last_seen_at),
  tgt.event_id = src.event_id,
  tgt._updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
  source_name, source_id, language_code, poi_id, poi_group_id,
  is_primary_source, match_method, source_status,
  first_seen_at, last_seen_at, event_id, _created_at, _updated_at
) VALUES (
  src.source_name, src.source_id, src.language_code, src.poi_id, src.poi_group_id,
  true, 'exact_id', 'active',
  src.first_seen_at, src.last_seen_at, src.event_id,
  current_timestamp(), current_timestamp()
)
""")

# ---------- 5. Sanity checks ----------
summary = spark.sql(f"""
SELECT
  COUNT(*) AS stage_rows,
  COUNT(DISTINCT poi_id) AS poi_count,
  COUNT(DISTINCT concat(source_name, '|', source_id)) AS source_poi_count,
  COUNT(DISTINCT lang) AS lang_count
FROM {STAGE_LANG}
WHERE run_id='{run_id}'
""").first()
print(summary)

# Invariant: same poi_id must not map to multiple languages.
invalid = spark.sql(f"""
SELECT poi_id, COUNT(DISTINCT lang) c
FROM {STAGE_LANG}
WHERE run_id='{run_id}'
GROUP BY poi_id HAVING c > 1
LIMIT 1
""").count()
if invalid:
    raise Exception("INVARIANT_FAILED: one poi_id mapped to multiple languages")

notebookutils.notebook.exit("SUCCESS")


In [ ]:
# # Fabric Notebook: NB_10_BRONZE_TO_SILVER_PREPROCESS
# # Default Lakehouse: lh_vv_silver (MANDATORY)
# # READ : Bronze poi_raw_event
# # WRITE: Silver _stage_raw_event, _stage_poi_language, poi_source_map
# # Core responsibility: one raw event -> N language-specific logical POIs.

# from pyspark.sql import functions as F
# from pyspark.sql.types import *

# # PARAMETERS - mark as Fabric parameter cell
# run_id = globals().get("run_id", "DEV_RUN")
# watermark_from = globals().get("watermark_from", "1901-01-01 00:00:00")
# watermark_to = globals().get("watermark_to", "2999-12-31 00:00:00")

# BRONZE_RAW = "sgr_visitvn_dev.lh_vv_bronze.dbo.poi_raw_event"
# STAGE_RAW = "_stage_raw_event"
# STAGE_LANG = "_stage_poi_language"
# SOURCE_MAP = "poi_source_map"

# # ---------- Helper: current deterministic ID formatting ----------
# def uuidish_md5_sql(expr: str) -> str:
#     return f"regexp_replace(md5({expr}), '(.{{8}})(.{{4}})(.{{4}})(.{{4}})(.{{12}})', '$1-$2-$3-$4-$5')"

# # ---------- 1. Create run-scoped staging tables ----------
# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {STAGE_RAW} (
#   run_id STRING,
#   event_id STRING,
#   source_name STRING,
#   source_id STRING,
#   crawled_at TIMESTAMP,
#   root_language_code STRING,
#   normalized_payload STRING,
#   poi_name STRING,
#   poi_name_normalized STRING,
#   business_sector STRING,
#   business_category STRING,
#   types_json STRING,
#   operating_status STRING,
#   slug STRING,
#   partner_id STRING,
#   address_json STRING,
#   contact_json STRING,
#   price_json STRING,
#   content_json STRING,
#   rating_json STRING,
#   review_json STRING,
#   media_json STRING,
#   opening_hours_json STRING,
#   policy_json STRING,
#   amenity_json STRING,
#   raw_data_json STRING,
#   enrichment_json STRING,
#   enrichment_siblings_json STRING,
#   created_at TIMESTAMP
# ) USING DELTA
# """)

# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {STAGE_LANG} (
#   run_id STRING,
#   event_id STRING,
#   source_name STRING,
#   source_id STRING,
#   crawled_at TIMESTAMP,
#   lang STRING,
#   poi_id STRING,
#   poi_group_id STRING,
#   is_root_language BOOLEAN,
#   poi_name STRING,
#   poi_name_normalized STRING,
#   business_sector STRING,
#   business_category STRING,
#   types_json STRING,
#   operating_status STRING,
#   slug STRING,
#   partner_id STRING,
#   address_json STRING,
#   contact_json STRING,
#   price_json STRING,
#   content_json STRING,
#   rating_json STRING,
#   review_json STRING,
#   media_json STRING,
#   opening_hours_json STRING,
#   policy_json STRING,
#   amenity_json STRING,
#   raw_data_json STRING,
#   localized_enrichment_json STRING,
#   source_enrichment_poi_id STRING,
#   enrichment_id STRING,
#   enrichment_lang STRING,
#   first_seen_at TIMESTAMP,
#   last_seen_at TIMESTAMP,
#   rn_by_poi INT,
#   created_at TIMESTAMP
# ) USING DELTA
# """)

# # Rerun safety for same run_id
# spark.sql(f"DELETE FROM {STAGE_RAW} WHERE run_id = '{run_id}'")
# spark.sql(f"DELETE FROM {STAGE_LANG} WHERE run_id = '{run_id}'")

# # ---------- 2. Stage raw: parse only top-level/scalar fields and preserve component JSON ----------
# spark.sql(f"""
# INSERT INTO {STAGE_RAW}
# SELECT
#   '{run_id}' AS run_id,
#   CAST(event_id AS STRING),
#   source_name,
#   source_id,
#   crawled_at,
#   lower(trim(language_code)) AS root_language_code,
#   normalized_payload,
#   get_json_object(normalized_payload, '$.poi_name') AS poi_name,
#   get_json_object(normalized_payload, '$.poi_name_normalized') AS poi_name_normalized,
#   get_json_object(normalized_payload, '$.business_sector') AS business_sector,
#   get_json_object(normalized_payload, '$.business_category') AS business_category,
#   get_json_object(normalized_payload, '$.types') AS types_json,
#   get_json_object(normalized_payload, '$.operating_status') AS operating_status,
#   get_json_object(normalized_payload, '$.slug') AS slug,
#   get_json_object(normalized_payload, '$.partner_id') AS partner_id,
#   get_json_object(normalized_payload, '$.poi_address') AS address_json,
#   get_json_object(normalized_payload, '$.poi_contact') AS contact_json,
#   get_json_object(normalized_payload, '$.poi_price') AS price_json,
#   get_json_object(normalized_payload, '$.poi_content') AS content_json,
#   get_json_object(normalized_payload, '$.poi_rating') AS rating_json,
#   get_json_object(normalized_payload, '$.poi_review') AS review_json,
#   get_json_object(normalized_payload, '$.poi_media') AS media_json,
#   get_json_object(normalized_payload, '$.poi_opening_hours') AS opening_hours_json,
#   get_json_object(normalized_payload, '$.policies') AS policy_json,
#   get_json_object(normalized_payload, '$.poi_amenity') AS amenity_json,
#   get_json_object(normalized_payload, '$.raw_data') AS raw_data_json,
#   get_json_object(normalized_payload, '$.extra_info.enrichment') AS enrichment_json,
#   get_json_object(normalized_payload, '$.extra_info.enrichmentSiblings') AS enrichment_siblings_json,
#   current_timestamp()
# FROM {BRONZE_RAW}
# WHERE normalized_payload IS NOT NULL
#   AND source_name IS NOT NULL
#   AND source_id IS NOT NULL
#   AND crawled_at > TIMESTAMP('{watermark_from}')
#   AND crawled_at <= TIMESTAMP('{watermark_to}')
# """)

# # ---------- 3. Expand dynamic languages ----------
# # Spark SQL MAP<STRING,STRING> lets us explode arbitrary sibling keys without hardcoding en/ko/...
# poi_id_expr = uuidish_md5_sql("concat(source_name, source_id, lang)")
# group_id_expr = uuidish_md5_sql("concat(source_name, source_id)")

# spark.sql(f"""
# INSERT INTO {STAGE_LANG}
# WITH base AS (
#   SELECT
#     *,
#     from_json(enrichment_siblings_json, 'MAP<STRING,STRING>') AS sibling_map,
#     lower(trim(get_json_object(enrichment_json, '$.lang'))) AS primary_enrichment_lang
#   FROM {STAGE_RAW}
#   WHERE run_id = '{run_id}'
# ),
# lang_rows AS (
#   -- root language
#   SELECT *, root_language_code AS lang,
#          CASE WHEN root_language_code = primary_enrichment_lang THEN enrichment_json ELSE element_at(sibling_map, root_language_code) END AS localized_enrichment_json
#   FROM base WHERE root_language_code IS NOT NULL AND root_language_code <> ''

#   UNION ALL

#   -- primary enrichment language (in case root language is missing/different)
#   SELECT *, primary_enrichment_lang AS lang, enrichment_json AS localized_enrichment_json
#   FROM base WHERE primary_enrichment_lang IS NOT NULL AND primary_enrichment_lang <> ''

#   UNION ALL

#   -- dynamic sibling languages
#   SELECT b.*, lower(trim(kv.key)) AS lang, kv.value AS localized_enrichment_json
#   FROM base b
#   LATERAL VIEW explode(b.sibling_map) kv AS key, value
# ),
# normalized_lang_rows AS (
#   SELECT
#     *,
#     lower(trim(lang)) AS normalized_lang
#   FROM lang_rows
#   WHERE lang IS NOT NULL AND trim(lang) <> ''
# ),
# event_language_validation AS (
#   SELECT
#     event_id,
#     source_name,
#     source_id,
#     MAX(CASE WHEN normalized_lang = 'vi' THEN 1 ELSE 0 END) AS has_vi,
#     MAX(CASE WHEN normalized_lang = 'en' THEN 1 ELSE 0 END) AS has_en,
#     MAX(CASE WHEN normalized_lang = 'ko' THEN 1 ELSE 0 END) AS has_ko
#   FROM normalized_lang_rows
#   GROUP BY event_id, source_name, source_id
# ),
# valid_events AS (
#   SELECT event_id, source_name, source_id
#   FROM event_language_validation
#   WHERE has_vi = 1
#     AND has_en = 1
#     AND has_ko = 1
# ),
# dedup_lang AS (
#   SELECT
#     nlr.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY nlr.event_id, nlr.source_name, nlr.source_id, nlr.normalized_lang
#       ORDER BY
#         CASE WHEN nlr.localized_enrichment_json IS NOT NULL THEN 0 ELSE 1 END,
#         nlr.crawled_at DESC
#     ) AS lang_rn
#   FROM normalized_lang_rows nlr
#   INNER JOIN valid_events ve
#     ON ve.event_id = nlr.event_id
#    AND ve.source_name = nlr.source_name
#    AND ve.source_id = nlr.source_id
# ),
# expanded AS (
#   SELECT
#     run_id, event_id, source_name, source_id, crawled_at, lang,
#     {poi_id_expr} AS poi_id,
#     {group_id_expr} AS poi_group_id,
#     (lang = root_language_code) AS is_root_language,
#     -- Prefer localized enrichment display fields; otherwise shared root fields.
#     COALESCE(get_json_object(localized_enrichment_json, '$.poiName'), poi_name) AS poi_name,
#     CASE
#       WHEN get_json_object(localized_enrichment_json, '$.poiName') IS NOT NULL
#         THEN lower(trim(get_json_object(localized_enrichment_json, '$.poiName')))
#       ELSE poi_name_normalized
#     END AS poi_name_normalized,
#     COALESCE(get_json_object(localized_enrichment_json, '$.refBusinessSector'), business_sector) AS business_sector,
#     business_category,
#     types_json,
#     operating_status,
#     slug,
#     partner_id,
#     address_json, contact_json, price_json, content_json, rating_json, review_json,
#     media_json, opening_hours_json, policy_json, amenity_json, raw_data_json,
#     localized_enrichment_json,
#     get_json_object(localized_enrichment_json, '$.poiId') AS source_enrichment_poi_id,
#     get_json_object(localized_enrichment_json, '$.enrichmentId') AS enrichment_id,
#     lower(trim(get_json_object(localized_enrichment_json, '$.lang'))) AS enrichment_lang,
#     crawled_at AS first_seen_at,
#     crawled_at AS last_seen_at
#   FROM dedup_lang
#   WHERE lang_rn = 1
#     AND lower(trim(lang)) IN ('vi','en','ko')
# ),
# ranked AS (
#   SELECT *,
#     MIN(crawled_at) OVER (PARTITION BY source_name, source_id, lang) AS batch_first_seen_at,
#     MAX(crawled_at) OVER (PARTITION BY source_name, source_id, lang) AS batch_last_seen_at,
#     ROW_NUMBER() OVER (PARTITION BY poi_id ORDER BY crawled_at DESC, event_id DESC) AS rn_by_poi
#   FROM expanded
# )
# SELECT
#   run_id, event_id, source_name, source_id, crawled_at, lang, poi_id, poi_group_id,
#   is_root_language, poi_name, poi_name_normalized, business_sector, business_category,
#   types_json, operating_status, slug, partner_id,
#   address_json, contact_json, price_json, content_json, rating_json, review_json, media_json,
#   opening_hours_json, policy_json, amenity_json, raw_data_json,
#   localized_enrichment_json, source_enrichment_poi_id, enrichment_id, enrichment_lang,
#   batch_first_seen_at, batch_last_seen_at, rn_by_poi, current_timestamp()
# FROM ranked
# """)

# # ---------- 3B. Required-language validation summary ----------
# # Business rule:
# #   A raw event is eligible only when it contains ALL required languages: vi, en, ko.
# #   Events missing any required language are skipped individually and DO NOT fail the batch.
# #   Only valid events were inserted into _stage_poi_language above.
# language_validation_summary = spark.sql(f"""
# WITH base AS (
#   SELECT
#     event_id,
#     source_name,
#     source_id,
#     lower(trim(root_language_code)) AS root_language_code,
#     lower(trim(get_json_object(enrichment_json, '$.lang'))) AS primary_enrichment_lang,
#     from_json(enrichment_siblings_json, 'MAP<STRING,STRING>') AS sibling_map
#   FROM {STAGE_RAW}
#   WHERE run_id = '{run_id}'
# ),
# langs AS (
#   SELECT event_id, source_name, source_id, root_language_code AS lang
#   FROM base
#   WHERE root_language_code IS NOT NULL AND root_language_code <> ''

#   UNION ALL

#   SELECT event_id, source_name, source_id, primary_enrichment_lang AS lang
#   FROM base
#   WHERE primary_enrichment_lang IS NOT NULL AND primary_enrichment_lang <> ''

#   UNION ALL

#   SELECT
#     b.event_id,
#     b.source_name,
#     b.source_id,
#     lower(trim(kv.key)) AS lang
#   FROM base b
#   LATERAL VIEW explode(b.sibling_map) kv AS key, value
# ),
# validation AS (
#   SELECT
#     event_id,
#     source_name,
#     source_id,
#     MAX(CASE WHEN lang = 'vi' THEN 1 ELSE 0 END) AS has_vi,
#     MAX(CASE WHEN lang = 'en' THEN 1 ELSE 0 END) AS has_en,
#     MAX(CASE WHEN lang = 'ko' THEN 1 ELSE 0 END) AS has_ko
#   FROM langs
#   WHERE lang IS NOT NULL AND lang <> ''
#   GROUP BY event_id, source_name, source_id
# )
# SELECT
#   COUNT(*) AS evaluated_event_count,
#   SUM(CASE WHEN has_vi = 1 AND has_en = 1 AND has_ko = 1 THEN 1 ELSE 0 END) AS valid_event_count,
#   SUM(CASE WHEN NOT (has_vi = 1 AND has_en = 1 AND has_ko = 1) THEN 1 ELSE 0 END) AS skipped_event_count
# FROM validation
# """).first()

# print(
#     "NB10 required-language validation:",
#     {
#         "evaluated_event_count": language_validation_summary["evaluated_event_count"],
#         "valid_event_count": language_validation_summary["valid_event_count"],
#         "skipped_event_count": language_validation_summary["skipped_event_count"],
#     },
# )

# # ---------- 4. Persistent source map MERGE: NEVER TRUNCATE ----------
# # Add poi_group_id if approved/needed. Safe to run once manually if ALTER conflicts on existing column.
# try:
#     spark.sql(f"ALTER TABLE {SOURCE_MAP} ADD COLUMNS (poi_group_id STRING)")
# except Exception:
#     pass

# spark.sql(f"""
# MERGE INTO {SOURCE_MAP} AS tgt
# USING (
#   SELECT
#     source_name,
#     source_id,
#     lang AS language_code,
#     poi_id,
#     poi_group_id,
#     MIN(first_seen_at) AS first_seen_at,
#     MAX(last_seen_at) AS last_seen_at,
#     max_by(event_id, crawled_at) AS event_id
#   FROM {STAGE_LANG}
#   WHERE run_id = '{run_id}'
#   GROUP BY source_name, source_id, lang, poi_id, poi_group_id
# ) src
# ON tgt.source_name = src.source_name
# AND tgt.source_id = src.source_id
# AND tgt.language_code = src.language_code
# WHEN MATCHED THEN UPDATE SET
#   tgt.poi_id = src.poi_id,
#   tgt.poi_group_id = src.poi_group_id,
#   tgt.last_seen_at = greatest(tgt.last_seen_at, src.last_seen_at),
#   tgt.event_id = src.event_id,
#   tgt._updated_at = current_timestamp()
# WHEN NOT MATCHED THEN INSERT (
#   source_name, source_id, language_code, poi_id, poi_group_id,
#   is_primary_source, match_method, source_status,
#   first_seen_at, last_seen_at, event_id, _created_at, _updated_at
# ) VALUES (
#   src.source_name, src.source_id, src.language_code, src.poi_id, src.poi_group_id,
#   true, 'exact_id', 'active',
#   src.first_seen_at, src.last_seen_at, src.event_id,
#   current_timestamp(), current_timestamp()
# )
# """)

# # ---------- 5. Sanity checks ----------
# summary = spark.sql(f"""
# SELECT
#   COUNT(*) AS stage_rows,
#   COUNT(DISTINCT poi_id) AS poi_count,
#   COUNT(DISTINCT concat(source_name, '|', source_id)) AS source_poi_count,
#   COUNT(DISTINCT lang) AS lang_count
# FROM {STAGE_LANG}
# WHERE run_id='{run_id}'
# """).first()
# print(summary)

# # Invariant: same poi_id must not map to multiple languages.
# invalid = spark.sql(f"""
# SELECT poi_id, COUNT(DISTINCT lang) c
# FROM {STAGE_LANG}
# WHERE run_id='{run_id}'
# GROUP BY poi_id HAVING c > 1
# LIMIT 1
# """).count()
# if invalid:
#     raise Exception("INVARIANT_FAILED: one poi_id mapped to multiple languages")

# notebookutils.notebook.exit("SUCCESS")
